Done By. Prateek Dixit (ENROLL. 23322018, BRANCH. BS-MS ECONOMICS)



#Problem Difficulty Prediction





#Imports And Dataset Loading

In [28]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import json
warnings.filterwarnings('ignore')

In [29]:
data = []
with open("/content/drive/MyDrive/problems_data.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))

df = pd.DataFrame(data)
print(df.head())


                       title  \
0                        Uuu   
1             House Building   
2             Mario or Luigi   
3             The Wire Ghost   
4  Barking Up The Wrong Tree   

                                         description  \
0  Unununium (Uuu) was the name of the chemical\n...   
1  A number of eccentrics from central New York h...   
2  Mario and Luigi are playing a game where they ...   
3  Žofka is bending a copper wire. She starts wit...   
4  Your dog Spot is let loose in the park. Well, ...   

                                   input_description  \
0  The input consists of one line with two intege...   
1  The input consists of $10$ test cases, which a...   
2                                                      
3  The first line contains two integers $L$ and $...   
4  The first line of input consists of two intege...   

                                  output_description  \
0  The output consists of $M$ lines where the $i$...   
1  Print $K$ lines wi

#Data Preprocessing

In [30]:
df.head()

,title,description,input_description,output_description,sample_io,problem_class,problem_score,url
0,Uuu,Unununium (Uuu) was the name of the chemical\n...,The input consists of one line with two intege...,The output consists of $M$ lines where the $i$...,"[{'input': '7 10', 'output': '1 2 2 3 1 3 3 4 ...",hard,9.7,https://open.kattis.com/problems/uuu
1,House Building,A number of eccentrics from central New York h...,"The input consists of $10$ test cases, which a...",Print $K$ lines with\n the positions of the...,"[{'input': '0 2 3 2 50 60 50 30 50 40', 'outpu...",hard,9.7,https://open.kattis.com/problems/husbygge
2,Mario or Luigi,Mario and Luigi are playing a game where they ...,,,"[{'input': '', 'output': ''}]",hard,9.6,https://open.kattis.com/problems/marioorluigi
3,The Wire Ghost,Žofka is bending a copper wire. She starts wit...,The first line contains two integers $L$ and $...,The output consists of a single line consistin...,"[{'input': '4 3 3 C 2 C 1 C', 'output': 'GHOST...",hard,9.6,https://open.kattis.com/problems/thewireghost
4,Barking Up The Wrong Tree,"Your dog Spot is let loose in the park. Well, ...",The first line of input consists of two intege...,Write a single line containing the length need...,"[{'input': '2 0 10 0 10 10', 'output': '14.14'...",hard,9.6,https://open.kattis.com/problems/barktree


In [31]:
df.isnull().sum()

,0
title,0
description,0
input_description,0
output_description,0
sample_io,0
problem_class,0
problem_score,0
url,0


In [32]:
df.describe()

,problem_score
count,4112.000000
mean,5.114689
std,2.177770
min,1.100000
25%,3.300000
50%,5.200000
75%,6.900000
max,9.700000


In [33]:
df["text"] = (
    df["title"].fillna("") + " " +
    df["description"].fillna("") + " " +
    df["input_description"].fillna("") + " " +
    df["output_description"].fillna("")
)


In [34]:
def text_length_features(texts):
    return [[
        len(t),                 # total characters
        len(t.split()),         # number of words
        t.count('\n')           # number of lines
    ] for t in texts]


In [35]:
MATH_SYMBOLS = ['<=', '>=', '=', '+', '-', '*', '/', '%', '^']
def math_symbol_features(texts):
    feats = []
    for t in texts:
        row = []
        for s in MATH_SYMBOLS:
            row.append(t.count(s))
        feats.append(row)
    return feats


In [36]:
KEYWORDS = [
    "graph", "tree", "dfs", "bfs", "dp", "dynamic programming",
    "binary search", "greedy", "math", "mod",
    "prime", "segment tree", "fenwick",
    "bitmask", "shortest path", "flow", "matching",
    "n log n", "constraints", "optimize"
]


In [37]:
def keyword_features(texts):
    feats = []
    for t in texts:
        t = t.lower()
        feats.append([t.count(k) for k in KEYWORDS])
    return feats


In [38]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
X_train, X_test, y_class_train, y_class_test = train_test_split(
    df["text"], df["problem_class"], test_size=0.2, random_state=42
)


#Vectorization and Singular Value Decomposition

In [39]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)


In [40]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=100, random_state=42)

X_train_svd = svd.fit_transform(X_train_tfidf)
X_test_svd = svd.transform(X_test_tfidf)


In [41]:
from scipy.sparse import hstack

X_train_final = hstack([X_train_tfidf, X_train_num])
X_test_final = hstack([X_test_tfidf, X_test_num])


In [42]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_num = np.hstack([
    text_length_features(X_train),
    math_symbol_features(X_train),
    keyword_features(X_train)
])

X_test_num = np.hstack([
    text_length_features(X_test),
    math_symbol_features(X_test),
    keyword_features(X_test)
])

X_train_num = scaler.fit_transform(X_train_num)
X_test_num = scaler.transform(X_test_num)


In [43]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

clf = LinearSVC(class_weight="balanced")
clf.fit(X_train_final, y_class_train)

y_pred = clf.predict(X_test_final)
print("Accuracy:", accuracy_score(y_class_test, y_pred))


Accuracy: 0.5261239368165249


#Trying Different Classifiers

In [44]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_svd, y_class_train)
y_pred = rf.predict(X_test_svd)

print("RF Accuracy:", accuracy_score(y_class_test, y_pred))


RF Accuracy: 0.511543134872418


In [45]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

# Encode target labels to numerical values
le = LabelEncoder()
y_class_train_encoded = le.fit_transform(y_class_train)
y_class_test_encoded = le.transform(y_class_test)

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softmax",
    num_class=len(le.classes_),
    eval_metric="mlogloss",
    random_state=42
)

xgb.fit(X_train_svd, y_class_train_encoded)
y_pred_encoded = xgb.predict(X_test_svd)



print("XGB Accuracy:", accuracy_score(y_class_test_encoded, y_pred_encoded))


XGB Accuracy: 0.5006075334143378


#Comparing Classifiers

In [46]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(
    class_weight="balanced",
    max_iter=2000
)

clf.fit(X_train_svd, y_class_train)


LogisticRegression(class_weight='balanced', max_iter=2000)

In [47]:
from sklearn.metrics import confusion_matrix, accuracy_score
import numpy as np

labels = ["easy", "medium", "hard"]

# Predict
y_pred_lr = clf.predict(X_test_svd)

# Confusion Matrix
cm_lr = confusion_matrix(y_class_test, y_pred_lr, labels=labels)

print("Logistic Regression – Confusion Matrix")
print(cm_lr)

print("\nClass-wise Accuracy (Logistic Regression):")
for i, label in enumerate(labels):
    acc = cm_lr[i, i] / cm_lr[i].sum() if cm_lr[i].sum() > 0 else 0
    print(f"{label}: {acc:.3f}")

print("\nOverall Accuracy (Logistic Regression):",
      accuracy_score(y_class_test, y_pred_lr))


Logistic Regression – Confusion Matrix
[[ 80  39  17]
 [ 70  92 100]
 [ 80 135 210]]

Class-wise Accuracy (Logistic Regression):
easy: 0.588
medium: 0.351
hard: 0.494

Overall Accuracy (Logistic Regression): 0.4641555285540705


In [48]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score
import numpy as np

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_svd, y_class_train)


RandomForestClassifier(class_weight='balanced', max_depth=20, n_estimators=300,
                       n_jobs=-1, random_state=42)

In [49]:
y_pred_rf = rf.predict(X_test_svd)


In [50]:
labels = ["easy", "medium", "hard"]

cm_rf = confusion_matrix(
    y_class_test,
    y_pred_rf,
    labels=labels
)

print("Random Forest – Confusion Matrix")
print(cm_rf)


Random Forest – Confusion Matrix
[[ 20  30  86]
 [ 11  35 216]
 [  5  54 366]]


In [51]:
print("\nClass-wise Accuracy (Random Forest):")
for i, label in enumerate(labels):
    acc = cm_rf[i, i] / cm_rf[i].sum() if cm_rf[i].sum() > 0 else 0
    print(f"{label}: {acc:.3f}")



Class-wise Accuracy (Random Forest):
easy: 0.147
medium: 0.134
hard: 0.861


In [52]:
print("\nOverall Accuracy (Random Forest):",
      accuracy_score(y_class_test, y_pred_rf))



Overall Accuracy (Random Forest): 0.511543134872418


In [53]:
from sklearn.metrics import f1_score

print("RF Macro F1:", f1_score(y_class_test, y_pred_rf, average="macro"))
print("LR Macro F1:", f1_score(y_class_test, y_pred_lr, average="macro"))


RF Macro F1: 0.36200051686660295
LR Macro F1: 0.4480513189093586


##**Because of class imbalance comparing by accuracy might be misleading, Hence we will compare different classifiers using macro-f1 score**

#Trying Different Regression Models

In [54]:
# text column already created
# df["text"] = title + description + input + output

X = df["text"]
y_score = df["problem_score"]   # REGRESSION TARGET


In [55]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_score,
    test_size=0.2,
    random_state=42
)


In [56]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)


In [57]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(
    n_components=100,
    random_state=42
)

X_train_svd = svd.fit_transform(X_train_tfidf)
X_test_svd = svd.transform(X_test_tfidf)



In [58]:
from sklearn.ensemble import RandomForestRegressor

rf_reg = RandomForestRegressor(
    n_estimators=400,
    max_depth=25,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_reg.fit(X_train_svd, y_train)


RandomForestRegressor(max_depth=25, min_samples_leaf=2, min_samples_split=5,
                      n_estimators=400, n_jobs=-1, random_state=42)

In [59]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

y_pred = rf_reg.predict(X_test_svd)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("MAE :", mae)
print("RMSE:", rmse)


MAE : 1.7732059279493884
RMSE: 2.100638930437112


In [60]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    random_state=42
)

gbr.fit(X_train_svd, y_train)

y_pred_gbr = gbr.predict(X_test_svd)

print("GBR MAE :", mean_absolute_error(y_test, y_pred_gbr))
print("GBR RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_gbr)))


GBR MAE : 1.7560140570261413
GBR RMSE: 2.0906863204141


In [61]:
def score_to_class(score):
    if score < 2.7:
        return "Easy"
    elif score < 5.5:
        return "Medium"
    else:
        return "Hard"


In [62]:
y_pred_score = gbr.predict(X_test_svd)

y_pred_class = [score_to_class(s) for s in y_pred_score]
y_true_class = [score_to_class(s) for s in y_test]

from sklearn.metrics import accuracy_score, confusion_matrix

print("Derived Class Accuracy:",
      accuracy_score(y_true_class, y_pred_class))

print(confusion_matrix(y_true_class, y_pred_class))


Derived Class Accuracy: 0.44835965978128794
[[  3   9 115]
 [  1 186 240]
 [  0  89 180]]


In [63]:
import numpy as np

b1 = np.percentile(y_pred_score, 33)
b2 = np.percentile(y_pred_score, 66)

def adaptive_class(score):
    if score < b1: return "Easy"
    elif score < b2: return "Medium"
    return "Hard"


In [70]:

reg = Ridge(alpha=1.0)
reg.fit(X_train_tfidf, y_train)

y_pred = reg.predict(X_test_tfidf)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))


MAE: 1.771560033461539
RMSE: 2.1113056696700263


#Using Sentence Transformer

In [64]:
pip install sentence-transformers

In [65]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

X_train_emb = model.encode(
    X_train.tolist(),
    batch_size=32,
    show_progress_bar=True
)


X_test_emb = model.encode(
    X_test.tolist(),
    batch_size=32,
    show_progress_bar=True
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/26 [00:00<?, ?it/s]

In [66]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

clf = LinearSVC(class_weight="balanced", C=1.0)
clf.fit(X_train_emb, y_class_train)

y_pred = clf.predict(X_test_emb)

print("Accuracy:", accuracy_score(y_class_test, y_pred))
print(classification_report(y_class_test, y_pred))


Accuracy: 0.49817739975698666
              precision    recall  f1-score   support

        easy       0.39      0.51      0.44       136
        hard       0.64      0.58      0.61       425
      medium       0.36      0.35      0.36       262

    accuracy                           0.50       823
   macro avg       0.46      0.48      0.47       823
weighted avg       0.51      0.50      0.50       823



In [67]:
print("LSVC_ST Macro F1:", f1_score(y_class_test, y_pred, average="macro"))


LSVC_ST Macro F1: 0.4690848993432019


#Export And  Deployment

In [69]:
import joblib
import os

# Create the models directory if it doesn't exist
if not os.path.exists('models'):
    os.makedirs('models')

joblib.dump(vectorizer, "models/vectorizer.pkl")
joblib.dump(svd, "models/svd.pkl")
joblib.dump(clf, "models/lr_model.pkl")

joblib.dump(tfidf, "models/tfidf_reg.pkl")
joblib.dump(reg, "models/ridge_reg.pkl")

['models/ridge_reg.pkl']